# 02 - RWGAN: WGAN-GP + one frozen reward (digit "5")

**Where this sits:** WGAN-GP -> **`RWGAN`** -> M-RWGAN. Same WGAN-GP core as
notebook 01, plus **one** frozen reward network: a small CNN that outputs
`P(digit == 5)`. The generator loss becomes a lambda-annealed blend of the
adversarial term and a reward term that is minimised when every generated image
reads as a 5.

> **Outcome up front:** with a single frozen differentiable reward this biases the
> output toward 5 but does **not** collapse onto it -- class 5 ends up the mode at
> ~25-35% of samples, not >90%. The closing **Result** cell explains why (the
> generator games `R5`'s blind spots) and lists the exact parameters behind that
> number.

## Reward wiring (mirrors `src/mrwgan/losses.py`)

```
L_G = lambda * ( -E[ critic(G(z)) ] )  +  (1 - lambda) * w5 * mean( (1 - R5(G(z)))^2 )
```

`w5 = REWARD_WEIGHT` is the paper's `beta` coefficient: the reward term is bounded
in `[0, 1]` while the WGAN critic term is not, so without `w5` (~8) the
adversarial gradient entirely swamps the reward. With it, the reward biases the
generator toward 5 -- class 5 becomes the mode of the output distribution
(~25-35% of samples), though not a full single-mode collapse; see the closing
**Result** cell. `src/mrwgan/losses.py` folds the same factor into
`generator_model`'s `loss_weights`.

`lambda` is annealed by an inline `LambdaSchedule` that mirrors the one in
`src/mrwgan/losses.py`: it starts at 1.0 (all-adversarial) and steps down by
0.025 every epoch once epoch 20 is reached, floored at 0.15 (hit around epoch
54). Early training is "learn to look real"; the ~45 epochs after the floor
shift weight onto "look like a 5".

The reward CNN `R5` is trained here (~2 epochs, ~99% val accuracy on a
class-balanced "is it a 5?" relabelling of MNIST) and then **frozen**
(`trainable = False`). A separate 10-class reference classifier is also trained,
only to label the generator's output for the digit histogram.

**Self-contained:** all blocks are inline; nothing is imported from `mrwgan`.

## Imports

In [ ]:
import os
import time

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__, "| executing eagerly:", tf.executing_eagerly())

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Configuration

In [ ]:
NB_TAG = "02_rwgan_single_reward"

EPOCHS = 100           # set to 2-3 for a quick smoke run
N_EVAL_SAMPLES = 2000  # generated images used for the digit-class histogram

BATCH_SIZE = 128
LATENT_DIM = 100
N_CRITIC = 5           # critic updates per generator update (WGAN-GP)
GP_WEIGHT = 10.0       # gradient-penalty coefficient

REWARD_WEIGHT = 8.0    # paper's "beta": lifts the bounded [0, 1] reward MSE to the
                       # magnitude of the unbounded WGAN critic term. GANNoC uses 3
                       # (effective 1-5); src/mrwgan/losses.py uses 10. Sweep 5-12
                       # if 8 over- / under-steers.

## Data -- MNIST scaled to [-1, 1]

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()


def preprocess(images):
    """uint8 [0, 255] -> float32 [-1, 1], shape (N, 28, 28, 1)."""
    images = images.astype("float32")
    images = (images - 127.5) / 127.5
    return images[..., np.newaxis]


x_train_img = preprocess(x_train)
x_test_img = preprocess(x_test)

train_ds = (
    tf.data.Dataset.from_tensor_slices(x_train_img)
    .shuffle(60_000, seed=SEED)
    .batch(BATCH_SIZE, drop_remainder=True)
    .repeat()
    .prefetch(tf.data.AUTOTUNE)
)
STEPS_PER_EPOCH = x_train_img.shape[0] // BATCH_SIZE
ds_iter = iter(train_ds)
print("train images:", x_train_img.shape, "| range:", float(x_train_img.min()), float(x_train_img.max()))
print("steps per epoch:", STEPS_PER_EPOCH)

## Reward network -- frozen "is this a 5?" CNN

`R5(x)` in `[0, 1]`. Trained on a class-balanced binary relabelling of MNIST,
then frozen. This is the only reward signal in this notebook.

In [ ]:
def make_reward_cnn(name):
    """Small binary CNN: P(image is the target digit). Sigmoid output in [0, 1]."""
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(28, 28, 1)),
            tf.keras.layers.Conv2D(32, 3, activation="relu"),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.Conv2D(64, 3, activation="relu"),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ],
        name=name,
    )


def balanced_binary(target_digit):
    """Class-balanced binary relabel of MNIST: all target-digit images + an equal
    random sample of non-target images, shuffled."""
    pos = np.where(y_train == target_digit)[0]
    neg = np.where(y_train != target_digit)[0]
    rng = np.random.default_rng(SEED + target_digit)
    neg = rng.choice(neg, size=len(pos), replace=False)
    idx = rng.permutation(np.concatenate([pos, neg]))
    labels = (y_train[idx] == target_digit).astype("float32")
    return x_train_img[idx], labels

In [ ]:
xb5, yb5 = balanced_binary(5)
reward_net_5 = make_reward_cnn("reward_is_5")
reward_net_5.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
reward_net_5.fit(xb5, yb5, epochs=2, batch_size=128, validation_split=0.1)
reward_net_5.trainable = False  # frozen for the rest of the notebook

## Reference classifier -- for the digit histogram only

In [ ]:
def make_classifier():
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(28, 28, 1)),
            tf.keras.layers.Conv2D(32, 3, activation="relu"),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.Conv2D(64, 3, activation="relu"),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(10, activation="softmax"),
        ],
        name="reference_classifier",
    )


# Reference 10-class classifier -- used ONLY to label generated samples for the
# digit histogram. Not part of the training loss.
ref_clf = make_classifier()
ref_clf.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
ref_clf.fit(x_train_img, y_train, validation_data=(x_test_img, y_test), epochs=2, batch_size=128)
ref_clf.trainable = False

## Models -- CNN generator + CNN critic

In [ ]:
def make_generator():
    """z(100) -> 7x7x256 -> 14x14x128 -> 28x28x64 -> 28x28x1 (tanh)."""
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(LATENT_DIM,)),
            tf.keras.layers.Dense(7 * 7 * 256, use_bias=False),
            tf.keras.layers.Reshape((7, 7, 256)),
            tf.keras.layers.Conv2DTranspose(128, 5, strides=2, padding="same", use_bias=False),  # -> 14x14
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),
            tf.keras.layers.Conv2DTranspose(64, 5, strides=2, padding="same", use_bias=False),   # -> 28x28
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),
            tf.keras.layers.Conv2D(1, 7, padding="same", activation="tanh"),
        ],
        name="generator",
    )


def make_critic():
    """CNN critic. No BatchNorm: the WGAN-GP gradient penalty is a per-sample
    constraint and BatchNorm mixes statistics across the batch. LayerNorm is safe."""
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(28, 28, 1)),
            tf.keras.layers.Conv2D(64, 5, strides=2, padding="same"),   # -> 14x14
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.LeakyReLU(0.2),
            tf.keras.layers.Conv2D(128, 5, strides=2, padding="same"),  # -> 7x7
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.LeakyReLU(0.2),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(1),  # linear score
        ],
        name="critic",
    )


generator = make_generator()
critic = make_critic()
generator.summary()
critic.summary()

## WGAN-GP core -- optimizers, gradient penalty, critic step

In [ ]:
g_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5, beta_2=0.9)
d_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5, beta_2=0.9)


def gradient_penalty(real, fake):
    """E[(||grad_x critic(x_hat)||_2 - 1)^2], x_hat = eps*real + (1-eps)*fake, eps ~ U[0, 1]."""
    batch = tf.shape(real)[0]
    eps = tf.random.uniform([batch, 1, 1, 1], 0.0, 1.0)
    x_hat = eps * real + (1.0 - eps) * fake
    with tf.GradientTape() as tape:
        tape.watch(x_hat)
        d_hat = critic(x_hat, training=True)
    grads = tape.gradient(d_hat, x_hat)
    norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1, 2, 3]) + 1e-12)
    return tf.reduce_mean((norm - 1.0) ** 2)


@tf.function  # autograph-traced eager code -- NOT tf.compat.v1 graph mode
def train_critic_step(real):
    z = tf.random.normal([tf.shape(real)[0], LATENT_DIM])
    fake = generator(z, training=True)
    with tf.GradientTape() as tape:
        d_real = critic(real, training=True)
        d_fake = critic(fake, training=True)
        wasserstein = tf.reduce_mean(d_fake) - tf.reduce_mean(d_real)
        gp = gradient_penalty(real, fake)
        d_loss = wasserstein + GP_WEIGHT * gp
    grads = tape.gradient(d_loss, critic.trainable_variables)
    d_optimizer.apply_gradients(zip(grads, critic.trainable_variables))
    return d_loss

## Sampling / plotting helpers

In [ ]:
fixed_noise = tf.random.normal([64, LATENT_DIM], seed=SEED)


def save_sample_grid(path, title, show=True):
    imgs = generator(fixed_noise, training=False).numpy()
    imgs = np.clip((imgs + 1.0) / 2.0, 0.0, 1.0)
    fig, axes = plt.subplots(8, 8, figsize=(8, 8))
    for i, ax in enumerate(axes.flat):
        ax.imshow(imgs[i, :, :, 0], cmap="gray")
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    if show:
        plt.show()
    plt.close(fig)


def generate_images(n, batch=500):
    out = []
    for i in range(0, n, batch):
        z = tf.random.normal([min(batch, n - i), LATENT_DIM])
        out.append(generator(z, training=False).numpy())
    return np.concatenate(out, axis=0)

## Lambda annealing schedule (mirrors `src/mrwgan/losses.py`)

In [ ]:
class LambdaSchedule:
    """Anneals the adversarial / reward balance -- mirrors `LambdaSchedule` in
    `src/mrwgan/losses.py`.

    LAMBDA starts at 1.0 (all-adversarial) and decays by `delta` every
    `every_n_epochs` epochs once `start_epoch` is reached, floored at `floor`.
    The adversarial term is scaled by LAMBDA, the reward term by (1 - LAMBDA),
    so training shifts from "look real" to "match the target reward".

    The repo class uses start_epoch=50 / every_n_epochs=10 / delta=-0.05 /
    floor=0.2 over a 300-epoch NoC run. Here the same shape is scaled to a
    ~100-epoch run: 20 epochs of clean WGAN-GP warm-up, then a smooth 1-epoch
    step of -0.025 until LAMBDA hits the 0.15 floor around epoch 54, leaving
    ~45 epochs where the reward term dominates the generator loss.
    """

    def __init__(self, initial=1.0, delta=-0.025, start_epoch=20, every_n_epochs=1, floor=0.15):
        self.value = float(initial)
        self.delta = delta
        self.start_epoch = start_epoch
        self.every_n_epochs = every_n_epochs
        self.floor = floor

    def step(self, epoch):
        if epoch >= self.start_epoch and epoch % self.every_n_epochs == 0 and self.value > self.floor:
            self.value = max(self.floor, self.value + self.delta)
        return self.value

## Generator step and training loop

`L_G = lambda * ( -E[ critic(G(z)) ] )  +  (1 - lambda) * w5 * mean( (1 - R5(G(z)))^2 )`

with `w5 = REWARD_WEIGHT` (the paper's `beta`), which scales the bounded `[0, 1]`
reward term up to the magnitude of the unbounded WGAN critic term -- without it
the adversarial gradient entirely swamps the reward. With it, the reward biases
the generator toward 5: class 5 becomes the mode of the output distribution
(~25-35% of samples), but not a full single-mode collapse -- see the closing
**Result** cell for why a frozen reward cannot force that. `lambda` is held in a
`tf.Variable` and reassigned once per epoch (so the `@tf.function` is not
retraced), annealed 1.0 -> 0.15 by `LambdaSchedule` starting at epoch 20.

In [ ]:
lam_var = tf.Variable(1.0, dtype=tf.float32)


@tf.function
def train_generator_step():
    z = tf.random.normal([BATCH_SIZE, LATENT_DIM])
    with tf.GradientTape() as tape:
        fake = generator(z, training=True)
        adv = -tf.reduce_mean(critic(fake, training=True))
        r5 = reward_net_5(fake, training=False)               # P(digit == 5), shape (B, 1)
        reward_term = tf.reduce_mean(tf.square(1.0 - r5))
        g_loss = lam_var * adv + (1.0 - lam_var) * REWARD_WEIGHT * reward_term
    grads = tape.gradient(g_loss, generator.trainable_variables)
    g_optimizer.apply_gradients(zip(grads, generator.trainable_variables))
    return g_loss, adv, reward_term


lam_sched = LambdaSchedule()
history = {"d_loss": [], "g_loss": [], "adv": [], "reward": [], "lambda": []}

for epoch in range(1, EPOCHS + 1):
    lam_var.assign(lam_sched.step(epoch))
    t0 = time.time()
    d_hist, g_hist, a_hist, r_hist = [], [], [], []
    for _ in range(STEPS_PER_EPOCH):
        for _ in range(N_CRITIC):
            d_hist.append(float(train_critic_step(next(ds_iter))))
        g_loss, adv, reward_term = train_generator_step()
        g_hist.append(float(g_loss))
        a_hist.append(float(adv))
        r_hist.append(float(reward_term))
    history["d_loss"].append(float(np.mean(d_hist)))
    history["g_loss"].append(float(np.mean(g_hist)))
    history["adv"].append(float(np.mean(a_hist)))
    history["reward"].append(float(np.mean(r_hist)))
    history["lambda"].append(float(lam_var.numpy()))
    print(
        f"epoch {epoch:3d}/{EPOCHS} | lambda {float(lam_var.numpy()):.2f} | "
        f"d {np.mean(d_hist):+.3f} | g {np.mean(g_hist):+.3f} | reward {np.mean(r_hist):.3f} | {time.time() - t0:.1f}s"
    )
    if epoch % 10 == 0 or epoch == EPOCHS:
        save_sample_grid(
            os.path.join(OUTPUT_DIR, f"{NB_TAG}_samples_epoch{epoch:03d}.png"),
            f"{NB_TAG} - epoch {epoch}",
            show=False,
        )

save_sample_grid(os.path.join(OUTPUT_DIR, f"{NB_TAG}_samples.png"), f"{NB_TAG} - final ({EPOCHS} epochs)")

## Predicted-digit histogram over generated samples

In [ ]:
gen = generate_images(N_EVAL_SAMPLES)
pred = ref_clf.predict(gen, batch_size=500, verbose=0).argmax(axis=1)
counts = np.bincount(pred, minlength=10)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(10), counts, color="#4C72B0")
ax.set_xticks(range(10))
ax.set_xlabel("predicted digit")
ax.set_ylabel(f"count (of {N_EVAL_SAMPLES})")
ax.set_title(f"{NB_TAG} - predicted digit classes of generated samples")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, f"{NB_TAG}_digit_hist.png"), dpi=110)
plt.show()
plt.close(fig)

print("class counts:", {i: int(c) for i, c in enumerate(counts)})
print(f"share predicted as 5: {counts[5] / counts.sum():.1%}")

## Lambda / reward trace

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(range(1, EPOCHS + 1), history["reward"], color="#C44E52", label="reward term")
ax1.set_xlabel("epoch")
ax1.set_ylabel("mean (1 - R5)^2", color="#C44E52")
ax2 = ax1.twinx()
ax2.plot(range(1, EPOCHS + 1), history["lambda"], color="#4C72B0", label="lambda")
ax2.set_ylabel("lambda", color="#4C72B0")
ax1.set_title(f"{NB_TAG} - reward term vs lambda")
fig.tight_layout()
plt.show()

## Result

**What actually happens (documented, not aspirational).** With the shipped
hyperparameters this does **not** collapse onto the digit 5. The histogram
(`output/02_rwgan_single_reward_digit_hist.png`) shows class 5 as the *mode* at
roughly **25-35%** of samples (`share predicted as 5` was 32% at `SEED = 42`),
with a broad tail over the other nine digits; the sample grid is a digit mixture,
not an all-5s grid.

**Why.** Once `lambda` reaches its 0.15 floor the reward term `mean((1 - R5)^2)`
carries ~85% of the generator loss, and the generator minimises it the cheap
way: by drifting *off* the real-image manifold into `R5`'s blind spots
(`R5 -> ~0.99`, critic loss collapses to about -5) rather than by drawing only
clean 5s. An independent classifier therefore still sees a spread of digits. A
single frozen, differentiable reward cannot force a >90% single-digit collapse
without this gaming; getting there needs a non-differentiable / RL reward, a
reward co-trained with the GAN, or an auxiliary-classifier term - all of which
change the loss form this notebook deliberately keeps.

**Parameters that produce the ~30% result above** (all set in the cells above):
`EPOCHS = 100`, `BATCH_SIZE = 128`, `LATENT_DIM = 100`, `N_CRITIC = 5`,
`GP_WEIGHT = 10`, `REWARD_WEIGHT = 8`; `LambdaSchedule(initial=1.0, delta=-0.025,
start_epoch=20, every_n_epochs=1, floor=0.15)`; `Adam(2e-4, beta_1=0.5,
beta_2=0.9)` for G and D; binary CNN reward `R5` frozen after training. This was
the best honest configuration found in a ~26-config sweep over `REWARD_WEIGHT`
[3-20], `lambda` floor [0-0.55], anneal shape, `LATENT_DIM` {100..512}, reward
architecture (binary CNN / ensemble / 10-way-softmax) and longer training - none
exceeded a sustained ~35% verified five-share.